<a href="https://colab.research.google.com/github/Arfa-Tariq/learning-ai-engineering/blob/main/projects/06-Function-Calling-%26-Data_Extraction/05_Applications_of_FunctionCalling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q groq

from groq import Groq
from google.colab import userdata
import inspect, json
import requests

GROQ_API_KEY = userdata.get('groq_key')
client = Groq(api_key=GROQ_API_KEY)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.3 MB/s eta 0:00:00


In [ ]:
def get_completion(prompt, model="llama-3.3-70b-versatile"):
  messages = [{'role':'system','content': "You are being used as a tool calling agent, just respond with the function call ONLY"},
              {"role": "user", "content": prompt}]
  response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0,)
  return response.choices[0].message.content

In [ ]:
def get_completion_no_func(prompt, model="llama-3.3-70b-versatile"):
  messages = [{"role": "user", "content": prompt}]
  response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0,)
  return response.choices[0].message.content

In [ ]:
def get_completion_from_messages(messages, tools,
                                 model="llama-3.3-70b-versatile",
                                 temperature=0, ):
    response = client.chat.completions.create(
        model=model,
        tools=tools,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.tool_calls[0]

In [ ]:
def build_prompt(function_list, user_query):
  f_prompt = ""
  for function in function_list:
    signature = inspect.signature(function)
    docstring = function.__doc__
    prompt= f''' Function:
def {function.__name__}{signature}
  """
  {docstring.strip()}
  """
     '''
    f_prompt += prompt
  f_prompt+= f"User Query: {user_query}<human_end>"
  return f_prompt

In [ ]:
question = "Hey, can you tell me more about this R1 thing that was announced by Rabbit? "

no_function_calling_prompt = \
f"""
<s> [INST] {question} [/INST]
"""
get_completion_no_func(no_function_calling_prompt)

"You're referring to the R1 announcement by Rabbit. Unfortunately, I don't have have specific information on what the R1 is, as my knowledge stopped in 2023 and I'm not aware of any recent announcements. However, I can suggest some possible ways to find more information about it.\n\nRabbit is likely a company or organization that has made an announcement about a product, service, or project called R1. To learn more, you can try checking the company's official website, social media, or press releases for information about the R1.\n\nIf you have any more context or details about the R1, such as what industry or field it relates to, I may be able to provide more general information or point you in the direction of resources that could be helpful. Alternatively, you can also try searching online for news articles or blogs that may have covered the announcement."

## Providing Up To Date Information

In [ ]:
def do_web_search(full_user_prompt : str, num_results : int = 5):
    API_URL = f'https://api.tavily.com/search'
    payload = \
    {
      "api_key": userdata.get('tavily_key'),
      "query": full_user_prompt,
      "search_depth": "basic",
      "include_answer": False,
      "include_images": False,
      "include_raw_content": False,
      "max_results": num_results,
      "include_domains": [],
      "exclude_domains": []
    }
    import requests
    response = requests.post(API_URL, json=payload)
    response = response.json()
    all_results = "\n\n".join(item["content"] for item in response["results"])
    return all_results

In [ ]:
prompt = f""" Function:
def do_web_search(full_user_prompt : str, num_results : int = 5):
    '''
    Searches the web for the user question.
    '''

Example:
User Query: What is the oldest capital in the world?
Call: do_web_search(full_user_prompt="oldest capital")

User Query: {question}<human_end>
"""
fc_result = get_completion(prompt)
print (fc_result)

do_web_search(full_user_prompt="Rabbit R1 announcement")


In [ ]:
result = eval(fc_result)

In [ ]:
full_prompt = \
f"""
<s> [INST]
{result}

Use the information above to answer the following question concisely.

Question:
{question} [/INST]
"""


In [ ]:
grounded_response = get_completion(full_prompt)

In [ ]:
print(grounded_response)

R1(pocket_ai_companion)


## Chatting With Your SQL Database

In [ ]:
from utils import create_random_database
create_random_database()

In [ ]:
question = "What is the most expensive item we currently sell?"

In [ ]:
from utils import execute_sql, query_raven

schema = \
"""
CREATE TABLE IF NOT EXISTS toys (
    id INTEGER PRIMARY KEY,
    name TEXT,
    price REAL
);
"""

raven_prompt = \
f'''
Function:
def execute_sql(sql_code : str):
  """
  Runs sql code for a company internal database
  """

Schema: {schema}
User Query: {question}
'''

output = query_raven(raven_prompt)
print (f"LLM's function call: {output}")
database_result = eval(output)

LLM's function call: execute_sql(sql_code='SELECT name, price FROM toys ORDER BY price DESC LIMIT 1')

In [ ]:
full_prompt = \
f"""
<s> [INST]
{database_result}

Use the information above to answer the following question concisely.

Question:
{question} [/INST]
"""
grounded_response = query_raven(full_prompt)
print (grounded_response)

##Safer Interactions With Databases

In [ ]:
import sqlite3
import random

# Internal database name setting
DB_NAME = 'toy_database.db'

# Connect to the database
def connect_db():
    return sqlite3.connect(DB_NAME)

In [ ]:
# List all toys
def list_all_toys():
    with connect_db() as conn:
        cursor = conn.execute('SELECT * FROM toys')
        return cursor.fetchall()

In [ ]:
# Find toy by name prefix
def find_toy_by_prefix(prefix):
    with connect_db() as conn:
        query = 'SELECT * FROM toys WHERE name LIKE ?'
        cursor = conn.execute(query, (prefix + '%',))
        return cursor.fetchall()

In [ ]:
# Find toys in a price range
def find_toys_in_price_range(low_price, high_price):
    with connect_db() as conn:
        query = 'SELECT * FROM toys WHERE price BETWEEN ? AND ?'
        cursor = conn.execute(query, (low_price, high_price))
        return cursor.fetchall()

In [ ]:
# Get a random selection of toys
def get_random_toys(count=5):
    with connect_db() as conn:
        cursor = conn.execute('SELECT * FROM toys')
        all_toys = cursor.fetchall()
        return random.sample(all_toys, min(count, len(all_toys)))
# Function to get the most expensive toy
def get_most_expensive_toy(count=1):
    with connect_db() as conn:
        cursor = conn.execute(f'SELECT * FROM toys ORDER BY price DESC LIMIT {count}')
        return cursor.fetchone()
# Function to get the cheapest toy
def get_cheapest_toy(count=1):
    with connect_db() as conn:
        cursor = conn.execute('SELECT * FROM toys ORDER BY price ASC LIMIT {count}')
        return cursor.fetchone()

In [ ]:
raven_prompt = \
f'''
Function:
def list_all_toys():
    """
    Retrieves a list of all toys from the database. This function does not take any parameters.
    Returns: A list of tuples, where each tuple represents a toy with all its attributes (id, name, price).
    """

Function:
def find_toy_by_prefix(prefix):
    """
    Searches for and retrieves toys whose names start with a specified prefix.
    Parameters:
    - prefix (str): The prefix to search for in toy names.
    Returns: A list of tuples, where each tuple represents a toy that matches the prefix criteria.
    """

Function:
def find_toys_in_price_range(low_price, high_price):
    """
    Finds and returns toys within a specified price range.
    Parameters:
    - low_price (float): The lower bound of the price range.
    - high_price (float): The upper bound of the price range.
    Returns: A list of tuples, each representing a toy whose price falls within the specified range.
    """

Function:
def get_random_toys():
    """
    Selects and returns a random set of toys from the database, simulating a "featured toys" list.

    Returns: A list of tuples, each representing a randomly selected toy. The number of toys returned is up to the specified count.
    """

Function:
def get_most_expensive_toy(count : int):
    """
    Retrieves the most expensive toy from the database.
    This function does not take any parameters.

    Returns: A tuple representing the most expensive toy, including its id, name, and price.
    """

Function:
def get_cheapest_toy(count : int):
    """
    Finds and retrieves the cheapest toy in the database.
    This function does not take any parameters.

    Returns: A tuple representing the cheapest toy, including its id, name, and price.
    """

User Query: {question}<human_end>

'''

output = query_raven(raven_prompt)
print (output)
results = eval(output)

get_most_expensive_toy(count=1)

In [ ]:
full_prompt = \
f"""
<s> [INST]
{database_result}

Use the information above to answer the following question in a single sentence.

Question:
{question} [/INST]
"""
grounded_response = query_raven(full_prompt)
print (grounded_response)